# 04 · XGBoost

Goal:

1. Reuse the exact chronological train, validation, and test design from Notebook 03.
2. Use **2024 only as an internal early-stopping period** to choose the number of boosting rounds.
3. Retrain XGBoost on all available 2018 to 2024 training data.
4. Evaluate once on the untouched 2025 validation set.
5. Compare directly against Logistic Regression.
6. Keep the 2026 test set untouched.

## Evaluation design

* 2018 to 2023: temporary fit set
* 2024: temporary early-stopping set
* 2018 to 2024: final XGBoost training after selecting `n_estimators`
* 2025: validation comparison
* 2026: untouched final test

In [ ]:
!pip install -q xgboost==3.4.1

In [ ]:
from google.colab import drive
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "metrics"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [REPORTS_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ML_DATA_FILE = PROCESSED_DATA_DIR / "ml_features.csv"
BASELINE_METRICS_FILE = (
    REPORTS_DIR / "03_baseline_validation_metrics.csv"
)
XGBOOST_METRICS_FILE = (
    REPORTS_DIR / "04_xgboost_validation_metrics.csv"
)
XGBOOST_MODEL_FILE = (
    MODEL_DIR / "xgboost_candidate.json"
)
XGBOOST_CONFIG_FILE = (
    MODEL_DIR / "xgboost_candidate_metadata.json"
)

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("Input:", ML_DATA_FILE)

## 1. Load and validate processed features

In [ ]:
if not ML_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {ML_DATA_FILE}. "
        "Run Notebook 02 first."
    )

ml_data = pd.read_csv(
    ML_DATA_FILE,
    parse_dates=["Date"],
)

ml_data = (
    ml_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Change_1D",
    "Relative_Volume_20D",
    "Price_vs_MA_5D",
    "Price_vs_MA_20D",
    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",
    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",
    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

CATEGORICAL_COLUMNS = ["Ticker"]
MODEL_COLUMNS = FEATURE_COLUMNS + CATEGORICAL_COLUMNS
TARGET_COLUMN = "Large_Move_5D"
TARGET_HORIZON_DAYS = 5

required_columns = ["Date", TARGET_COLUMN] + MODEL_COLUMNS
missing_columns = set(required_columns) - set(ml_data.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if ml_data[MODEL_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Missing model values detected.")

if np.isinf(ml_data[FEATURE_COLUMNS].to_numpy()).any():
    raise ValueError("Infinite numeric features detected.")

print("Shape:", ml_data.shape)
print(
    "Date range:",
    ml_data["Date"].min().date(),
    "to",
    ml_data["Date"].max().date(),
)
print("Feature validation passed.")

## 2. Recreate the exact Notebook 03 chronological split

In [ ]:
VALIDATION_START = pd.Timestamp("2025-01-01")
TEST_START = pd.Timestamp("2026-01-01")

train_data = ml_data[
    ml_data["Date"] < VALIDATION_START
].copy()

validation_data = ml_data[
    (ml_data["Date"] >= VALIDATION_START)
    & (ml_data["Date"] < TEST_START)
].copy()

test_data = ml_data[
    ml_data["Date"] >= TEST_START
].copy()


def purge_last_trading_dates(data, n_dates):
    unique_dates = np.array(sorted(data["Date"].unique()))

    if len(unique_dates) <= n_dates:
        raise ValueError("Not enough dates to apply purge.")

    purged_dates = unique_dates[-n_dates:]

    cleaned = data[
        ~data["Date"].isin(purged_dates)
    ].copy()

    return cleaned, purged_dates


train_data, purged_train_dates = purge_last_trading_dates(
    train_data,
    TARGET_HORIZON_DAYS,
)

validation_data, purged_validation_dates = purge_last_trading_dates(
    validation_data,
    TARGET_HORIZON_DAYS,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [
            len(train_data),
            len(validation_data),
            len(test_data),
        ],
        "start_date": [
            train_data["Date"].min(),
            validation_data["Date"].min(),
            test_data["Date"].min(),
        ],
        "end_date": [
            train_data["Date"].max(),
            validation_data["Date"].max(),
            test_data["Date"].max(),
        ],
        "positive_rate": [
            train_data[TARGET_COLUMN].mean() * 100,
            validation_data[TARGET_COLUMN].mean() * 100,
            test_data[TARGET_COLUMN].mean() * 100,
        ],
    }
)

if not (
    train_data["Date"].max()
    < validation_data["Date"].min()
    < test_data["Date"].min()
):
    raise ValueError("Chronological ordering failed.")

split_summary

## 3. Internal early-stopping split

2024 is used only to choose the number of boosting rounds. The final 2025 validation period is not used for early stopping.

In [ ]:
EARLY_STOP_START = pd.Timestamp("2024-01-01")

temporary_fit_data = train_data[
    train_data["Date"] < EARLY_STOP_START
].copy()

early_stop_data = train_data[
    train_data["Date"] >= EARLY_STOP_START
].copy()

temporary_fit_data, internal_purged_dates = (
    purge_last_trading_dates(
        temporary_fit_data,
        TARGET_HORIZON_DAYS,
    )
)

internal_summary = pd.DataFrame(
    {
        "split": ["temporary_fit", "early_stop"],
        "rows": [
            len(temporary_fit_data),
            len(early_stop_data),
        ],
        "start_date": [
            temporary_fit_data["Date"].min(),
            early_stop_data["Date"].min(),
        ],
        "end_date": [
            temporary_fit_data["Date"].max(),
            early_stop_data["Date"].max(),
        ],
        "positive_rate": [
            temporary_fit_data[TARGET_COLUMN].mean() * 100,
            early_stop_data[TARGET_COLUMN].mean() * 100,
        ],
    }
)

if temporary_fit_data["Date"].max() >= early_stop_data["Date"].min():
    raise ValueError("Internal early-stopping split overlaps.")

print(
    "Internal purged dates:",
    pd.to_datetime(internal_purged_dates).date,
)

internal_summary

## 4. Encode ticker

Numeric features are passed through unchanged because tree-based models do not require feature scaling.

In [ ]:
early_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            FEATURE_COLUMNS,
        ),
        (
            "ticker",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    sparse_threshold=0,
)

X_temp_fit = temporary_fit_data[MODEL_COLUMNS].copy()
y_temp_fit = temporary_fit_data[TARGET_COLUMN].astype(int).copy()

X_early_stop = early_stop_data[MODEL_COLUMNS].copy()
y_early_stop = early_stop_data[TARGET_COLUMN].astype(int).copy()

X_temp_fit_encoded = early_preprocessor.fit_transform(
    X_temp_fit
)

X_early_stop_encoded = early_preprocessor.transform(
    X_early_stop
)

print(
    "Temporary fit encoded shape:",
    X_temp_fit_encoded.shape,
)
print(
    "Early-stop encoded shape:",
    X_early_stop_encoded.shape,
)

## 5. Early stopping

We begin with many possible boosting rounds and let 2024 validation log loss determine when training should stop.

In [ ]:
XGB_PARAMS = {
    "objective": "binary:logistic",
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "tree_method": "hist",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
}

early_stop_model = xgb.XGBClassifier(
    **XGB_PARAMS,
    n_estimators=2000,
    early_stopping_rounds=50,
)

early_stop_model.fit(
    X_temp_fit_encoded,
    y_temp_fit,
    eval_set=[
        (
            X_early_stop_encoded,
            y_early_stop,
        )
    ],
    verbose=False,
)

BEST_N_ESTIMATORS = early_stop_model.best_iteration + 1

print("Best boosting rounds:", BEST_N_ESTIMATORS)
print(
    "Best early-stop log loss:",
    early_stop_model.best_score,
)

## 6. Retrain on all 2018 to 2024 training data

In [ ]:
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            FEATURE_COLUMNS,
        ),
        (
            "ticker",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    sparse_threshold=0,
)

X_train = train_data[MODEL_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].astype(int).copy()

X_validation = validation_data[MODEL_COLUMNS].copy()
y_validation = validation_data[TARGET_COLUMN].astype(int).copy()

# Prepared but deliberately not evaluated.
X_test = test_data[MODEL_COLUMNS].copy()
y_test = test_data[TARGET_COLUMN].astype(int).copy()

X_train_encoded = final_preprocessor.fit_transform(
    X_train
)

X_validation_encoded = final_preprocessor.transform(
    X_validation
)

final_feature_names = (
    final_preprocessor.get_feature_names_out()
)

xgb_model = xgb.XGBClassifier(
    **XGB_PARAMS,
    n_estimators=BEST_N_ESTIMATORS,
)

xgb_model.fit(
    X_train_encoded,
    y_train,
    verbose=False,
)

print("Final train shape:", X_train_encoded.shape)
print(
    "Final validation shape:",
    X_validation_encoded.shape,
)
print("Final XGBoost fitted.")

## 7. Evaluate 2025 validation

In [ ]:
def evaluate_classifier(name, model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(
            y,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            y,
            probabilities,
        ),
    }

    return metrics, predictions, probabilities


xgb_metrics, xgb_predictions, xgb_probabilities = (
    evaluate_classifier(
        "XGBoost",
        xgb_model,
        X_validation_encoded,
        y_validation,
    )
)

xgb_results = pd.DataFrame(
    [xgb_metrics]
).set_index("model")

xgb_results.round(4)

## 8. Compare with Notebook 03

In [ ]:
if not BASELINE_METRICS_FILE.exists():
    raise FileNotFoundError(
        "Notebook 03 metrics not found. "
        "Run Notebook 03 first."
    )

baseline_results = pd.read_csv(
    BASELINE_METRICS_FILE,
    index_col=0,
)

model_comparison = pd.concat(
    [baseline_results, xgb_results],
    axis=0,
)

display(model_comparison.round(4))

logistic_auc = model_comparison.loc[
    "LogisticRegression",
    "roc_auc",
]

xgb_auc = model_comparison.loc[
    "XGBoost",
    "roc_auc",
]

print(
    "XGBoost ROC AUC change vs Logistic Regression:",
    round(xgb_auc - logistic_auc, 4),
)

## 9. Classification report and confusion matrix

In [ ]:
print(
    classification_report(
        y_validation,
        xgb_predictions,
        target_names=[
            "Normal Move",
            "Large Move",
        ],
        digits=4,
    )
)

cm = confusion_matrix(
    y_validation,
    xgb_predictions,
)

print("Confusion matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Normal Move",
        "Large Move",
    ],
)

disp.plot()
plt.title("XGBoost - 2025 Validation")
plt.show()

## 10. Gain-based feature importance

This is a useful first diagnostic, but it is not causal explanation. SHAP will be introduced later.

In [ ]:
booster = xgb_model.get_booster()

gain_scores = booster.get_score(
    importance_type="gain"
)

importance_rows = []

for index, feature_name in enumerate(
    final_feature_names
):
    key = f"f{index}"

    importance_rows.append(
        {
            "feature": feature_name,
            "gain": gain_scores.get(
                key,
                0.0,
            ),
        }
    )

feature_importance = (
    pd.DataFrame(importance_rows)
    .sort_values(
        "gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_importance.head(15)

## 11. Save candidate artifacts

This is still a candidate model. The 2026 test set remains untouched.

In [ ]:
model_comparison.to_csv(
    XGBOOST_METRICS_FILE
)

xgb_model.save_model(
    XGBOOST_MODEL_FILE
)

candidate_metadata = {
    "xgboost_version": xgb.__version__,
    "best_n_estimators": int(
        BEST_N_ESTIMATORS
    ),
    "parameters": XGB_PARAMS,
    "training_start": str(
        train_data["Date"].min().date()
    ),
    "training_end": str(
        train_data["Date"].max().date()
    ),
    "validation_start": str(
        validation_data["Date"].min().date()
    ),
    "validation_end": str(
        validation_data["Date"].max().date()
    ),
    "test_evaluated": False,
}

with open(
    XGBOOST_CONFIG_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        candidate_metadata,
        file,
        indent=2,
    )

for file_path in [
    XGBOOST_METRICS_FILE,
    XGBOOST_MODEL_FILE,
    XGBOOST_CONFIG_FILE,
]:
    print(
        file_path.name,
        "exists:",
        file_path.exists(),
        "size:",
        file_path.stat().st_size
        if file_path.exists()
        else None,
    )

In [ ]:
reloaded_comparison = pd.read_csv(
    XGBOOST_METRICS_FILE,
    index_col=0,
)

if reloaded_comparison.shape != model_comparison.shape:
    raise ValueError(
        "Reloaded model comparison has wrong shape."
    )

reloaded_booster = xgb.XGBClassifier()
reloaded_booster.load_model(
    XGBOOST_MODEL_FILE
)

reload_predictions = reloaded_booster.predict(
    X_validation_encoded
)

if not np.array_equal(
    reload_predictions,
    xgb_predictions,
):
    raise ValueError(
        "Reloaded XGBoost model predictions differ."
    )

print("XGBoost pipeline completed successfully.")
print("2026 test set remains untouched.")

# What to send back

Send:

1. `internal_summary`
2. Best boosting rounds and best early-stop log loss
3. `model_comparison`
4. XGBoost classification report
5. Confusion matrix values
6. Top 15 feature importance rows

We will decide whether XGBoost is a genuine improvement before moving into PyTorch.